In [1]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import os
from matplotlib import pyplot as plt
from matplotlib.widgets import RectangleSelector
from matplotlib.patches import Rectangle
from pathlib import Path
import pdb

# Coordinates will be stored in the raw data's folder (parent folder) in Raw_data
# COORD_FOLDER = '/n/holylabs/LABS/gershman_lab/Users/Kelso/Raw_data'   # Holylabs location

COORD_FOLDER = '/n/netscratch/gershman_lab/Lab/zkelso/Regions_files'



# File we're playing with now is from 
#   /n/holylabs/LABS/gershman_lab/Users/Kelso/Raw_data/2025_09_19_13_53_26_trial_1_TC/dat.bin

class BinFrameSelector:
    def __init__(self):
        # Initialize instance variables
        self.frame = None
        self.regions = []
        self.fig = None
        self.ax = None
        self.selector = None
        self.rectangles = []
        self.current_bin_path = None  # Initialize the path variable
        
        # Create output widget for matplotlib plots
        self.plot_output = widgets.Output()
        
        # Create widgets
        self.file_path = widgets.Text(
            description='Bin File:',
            placeholder='Enter full path to experimental video file (dat.bin)',
            style={'description_width': 'initial'},
            layout={'width': '50%'}
        )
        self.frame_height = widgets.IntText(
            value=2048,
            description='Frame Height:',
            style={'description_width': 'initial'}
        )
        self.frame_width = widgets.IntText(
            value=2048,
            description='Frame Width:',
            style={'description_width': 'initial'}
        )
        self.load_button = widgets.Button(description='Load Frame')
        self.save_button = widgets.Button(description='Save Regions')
        self.clear_button = widgets.Button(description='Clear Regions')
        self.status = widgets.HTML(value='')
        
        # Layout - Include the plot output widget
        self.widget_box = widgets.VBox([
            widgets.HBox([self.file_path]),
            widgets.HBox([self.frame_width, self.frame_height]),
            widgets.HBox([self.load_button, self.save_button, self.clear_button]),
            self.status,
            self.plot_output  # Added plot output widget
        ])
        
        # Connect callbacks
        self.load_button.on_click(self.load_frame)
        self.save_button.on_click(self.save_regions)
        self.clear_button.on_click(self.clear_regions)

    def line_select_callback(self, eclick, erelease):
        """Callback for line selection"""
        if eclick.xdata is None or erelease.xdata is None:
            return
            
        x1, y1 = int(eclick.xdata), int(eclick.ydata)
        x2, y2 = int(erelease.xdata), int(erelease.ydata)
        
        # Ensure coordinates are in the correct order (top-left to bottom-right)
        x_min, x_max = min(x1, x2), max(x1, x2)
        y_min, y_max = min(y1, y2), max(y1, y2)
        
        # Add region to list
        self.regions.append([x_min, y_min, x_max, y_max])
        
        # Create and add rectangle patch
        width = x_max - x_min
        height = y_max - y_min
        rect = Rectangle((x_min, y_min), width, height,
                        fill=False, edgecolor='red', linewidth=2)
        self.ax.add_patch(rect)
        self.rectangles.append(rect)
        
        # Update status and redraw
        self.status.value = f'<p>Region {len(self.regions)} added: ({x_min}, {y_min}) to ({x_max}, {y_max})</p>'
        self.fig.canvas.draw_idle()
    
    def load_frame(self, _):
        try:
            filepath = Path(self.file_path.value.strip())
            if not filepath.exists():
                raise FileNotFoundError(f"File not found: {filepath}")
            
            # Store the path for later use
            self.current_bin_path = filepath
            
            # Read first frame from bin file
            frame_size = self.frame_width.value * self.frame_height.value
            with open(filepath, 'rb') as f:
                data = np.fromfile(f, dtype=np.uint8, count=frame_size)
            
            if len(data) < frame_size:
                raise ValueError(f"File too small for specified dimensions. Expected {frame_size} bytes, got {len(data)} bytes")
            
            # Reshape to 2D array
            self.frame = data.reshape((self.frame_height.value, self.frame_width.value))
            
            self.show_frame()
            self.status.value = f'<p style="color: green;">Frame loaded successfully from {filepath}</p>'
            
        except Exception as e:
            self.current_bin_path = None  # Reset path if load fails
            self.status.value = f'<p style="color: red;">Error loading frame: {str(e)}</p>'
            print(f"Detailed error: {e}")
    
    def show_frame(self):
        if self.frame is None:
            return
        
        # Clear the output widget
        self.plot_output.clear_output(wait=True)
        
        # Clear previous plot
        if self.fig is not None:
            plt.close(self.fig)
        
        # Reset regions and rectangles
        self.regions = []
        self.rectangles = []
        
        # Use the output widget context to display the plot
        with self.plot_output:
            # Create new plot
            self.fig, self.ax = plt.subplots(figsize=(10, 10))
            self.ax.imshow(self.frame, cmap='gray')
            self.ax.set_title('Click and drag to select regions\nPress Clear Regions to start over')
            
            # Enable selector
            self.selector = RectangleSelector(
                self.ax,
                self.line_select_callback,
                useblit=True,
                button=[1],
                minspanx=5,
                minspany=5,
                spancoords='pixels',
                interactive=True
            )
            
            plt.show()
    
    def save_regions(self, _):
        if not self.regions:
            self.status.value = '<p style="color: red;">No regions selected</p>'
            return
        
        if self.frame is None:
            self.status.value = '<p style="color: red;">No frame loaded</p>'
            return
            
        if self.current_bin_path is None:
            self.status.value = '<p style="color: red;">No bin file path available</p>'
            return
        
        try:
            # Convert regions to numpy array
            regions_array = np.array(self.regions)
            
            # Get parent folder name (immediate parent)
            parent_folder_name = self.current_bin_path.parent.name

            # REVERSION NOTE: Original server backup path was:
            # save_path_2 = os.path.join('/n/holylabs/LABS/gershman_lab/Users/madeleinesnyder/planaria_raw_video/', parent_folder_name, filename)
            # Directory creation check was:
            # if not os.path.exists(os.path.join('/n/holylabs/LABS/gershman_lab/Users/madeleinesnyder/planaria_raw_video/', parent_folder_name)):
            #     os.makedirs(os.path.join('/n/holylabs/LABS/gershman_lab/Users/madeleinesnyder/planaria_raw_video/', parent_folder_name))

            # Create filename in same directory as bin file
            h, w = self.frame.shape
            filename = f'regions_{parent_folder_name}_{w}x{h}.npy'
            save_path = self.current_bin_path.parent / filename

            # Old inconsistent saving of regions
            # # ===== DUMMY_DIRECTORY MODIFICATION START =====
            # # Original backup path was: save_path_2 = os.path.join('/Users/zacharykelso/Desktop', parent_folder_name, filename)
            # # Modified for Dummy_Directory testing:
            # save_path_2 = os.path.join('/Users/zacharykelso/Desktop/Dummy_Directory', parent_folder_name, filename)

            # # Original directory check was: if not os.path.exists(os.path.join('/Users/zacharykelso/Desktop', parent_folder_name)):
            # # Modified for Dummy_Directory testing:
            # if not os.path.exists(os.path.join('/Users/zacharykelso/Desktop/Dummy_Directory', parent_folder_name)):
            #     os.makedirs(os.path.join('/Users/zacharykelso/Desktop/Dummy_Directory', parent_folder_name))
            # # ===== DUMMY_DIRECTORY MODIFICATION END =====

            # Updated region saving
            # ===== DUMMY_DIRECTORY MODIFICATION START =====
            # Original backup path was: save_path_2 = os.path.join('/Users/zacharykelso/Desktop', parent_folder_name, filename)
            # Modified for Dummy_Directory testing to save to COORD_FOLDER:
            # COORD_FOLDER = '/Users/zacharykelso/Desktop/Dummy_Directory/FakeCoordFolder'
            # save_path_2 = os.path.join(COORD_FOLDER, parent_folder_name, filename)
            
            # Original directory check was: if not os.path.exists(os.path.join('/Users/zacharykelso/Desktop', parent_folder_name)):
            # Modified for Dummy_Directory testing to use COORD_FOLDER:
#             if not os.path.exists(os.path.join(COORD_FOLDER, parent_folder_name)):
#                 os.makedirs(os.path.join(COORD_FOLDER, parent_folder_name))
            # ===== DUMMY_DIRECTORY MODIFICATION END =====
            
            # np.save(save_path_2, regions_array, allow_pickle=False)
            np.save(save_path, regions_array, allow_pickle=False)
            #print(f"Saved {len(self.regions)} regions to {save_path_2} and {save_path}:")
            print(f"Saved {len(self.regions)} regions to {save_path}:")

            for i, region in enumerate(self.regions, 1):
                print(f"Region {i}: ({region[0]}, {region[1]}) to ({region[2]}, {region[3]})")
            self.status.value = f'<p style="color: green;">Saved {len(self.regions)} regions to {save_path}</p>'
            
        except Exception as e:
            self.status.value = f'<p style="color: red;">Error saving regions: {str(e)}</p>'
            print(f"Detailed error: {e}")
    
    def clear_regions(self, _):
        if self.frame is not None:
            # Remove all rectangle patches
            for rect in self.rectangles:
                rect.remove()
            self.rectangles = []
            self.regions = []
            self.fig.canvas.draw_idle()
            self.status.value = '<p>Regions cleared</p>'
    
    def display(self):
        display(self.widget_box)

# Enable interactive mode - CHANGED from %matplotlib notebook to %matplotlib ipympl
%matplotlib ipympl

# Create and display widget
print("Click and drag from the upper left corner to the lower right corner of the desired well.")
selector = BinFrameSelector()
selector.display()

Click and drag from the upper left corner to the lower right corner of the desired well.


Saved 6 regions to /n/netscratch/gershman_lab/Lab/zkelso/Raw_data/2025_10_17_14_18_23_trial_1_TC/regions_2025_10_17_14_18_23_trial_1_TC_2048x2048.npy:
Region 1: (155, 457) to (378, 1390)
Region 2: (394, 441) to (615, 1385)
Region 3: (642, 422) to (873, 1369)
Region 4: (878, 422) to (1110, 1372)
Region 5: (1102, 417) to (1362, 1353)
Region 6: (1357, 409) to (1591, 1361)
